# 07 - Optuna Hyperparameter Tuning

Bayesian hyperparameter optimisation for the top-3 regression models identified in notebook 04:

| Model | Library | Trials |
|-------|---------|--------|
| LightGBM | lightgbm | 50 |
| CatBoost | catboost | 50 |
| XGBoost | xgboost | 50 |

We use Optuna's TPE sampler to maximise R² on the validation set, then retrain on train+val and evaluate on the held-out test set.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import joblib
import time
import warnings
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from config import (DATA_PROCESSED, FIGURES_DIR, MODELS_DIR, RANDOM_STATE,
                     N_OPTUNA_TRIALS, EARLY_STOPPING_ROUNDS, MODELS_TUNED)
from src.evaluation.metrics import regression_metrics

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.makedirs(MODELS_TUNED, exist_ok=True)

## 1. Load Processed Data

In [ ]:
df_train = pd.read_parquet(DATA_PROCESSED / 'train.parquet')
df_val = pd.read_parquet(DATA_PROCESSED / 'val.parquet')
df_test = pd.read_parquet(DATA_PROCESSED / 'test.parquet')

print(f"Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")

# Define feature columns (exclude identifiers, target, and derived columns)
exclude_cols = ['City', 'Date', 'AQI', 'AQI_Bucket', 'season',
                'AQI_Calculated', 'AQI_Bucket_Calculated', 'Dominant_Pollutant']
feature_cols = [c for c in df_train.columns if c not in exclude_cols]
target_col = 'AQI'

print(f"Features: {len(feature_cols)}")
print(f"Target: {target_col}")

# Prepare arrays
X_train = df_train[feature_cols].values
y_train = df_train[target_col].values
X_val = df_val[feature_cols].values
y_val = df_val[target_col].values
X_test = df_test[feature_cols].values
y_test = df_test[target_col].values

# Handle any remaining NaN (fill with 0 for tree models)
X_train = np.nan_to_num(X_train, nan=0.0)
X_val = np.nan_to_num(X_val, nan=0.0)
X_test = np.nan_to_num(X_test, nan=0.0)

# Combined train+val for final retraining
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"X_trainval: {X_trainval.shape}, y_trainval: {y_trainval.shape}")

## 2. Define Optuna Objectives

In [ ]:
def objective_lightgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbosity': -1,
    }
    model = LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
    )
    y_pred = model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred)
    return metrics['R2']


def objective_xgboost(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    }
    model = XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    y_pred = model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred)
    return metrics['R2']


def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 10.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        'random_seed': RANDOM_STATE,
        'verbose': 0,
    }
    model = CatBoostRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )
    y_pred = model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred)
    return metrics['R2']

## 3. Run Optuna Studies

In [ ]:
studies = {}

study_configs = {
    'LightGBM': objective_lightgbm,
    'XGBoost': objective_xgboost,
    'CatBoost': objective_catboost,
}

for name, objective_fn in study_configs.items():
    print(f"\n{'='*60}")
    print(f"Optimising {name} ({N_OPTUNA_TRIALS} trials)...")
    print(f"{'='*60}")
    
    start = time.time()
    study = optuna.create_study(direction='maximize',
                                study_name=f'{name}_tuning',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective_fn, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    elapsed = time.time() - start
    
    studies[name] = study
    print(f"\n{name} completed in {elapsed:.1f}s")
    print(f"  Best R² (val): {study.best_value:.4f}")
    print(f"  Best trial:    #{study.best_trial.number}")

## 4. Best Parameters

In [ ]:
for name, study in studies.items():
    print(f"\n{'='*60}")
    print(f"{name} - Best Parameters (Trial #{study.best_trial.number})")
    print(f"{'='*60}")
    print(f"Best Validation R²: {study.best_value:.4f}")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

## 5. Retrain on Train+Val & Evaluate on Test Set

In [ ]:
final_results = {}
final_models = {}

# --- LightGBM ---
best_params = studies['LightGBM'].best_params.copy()
best_params.update({'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': -1})
lgbm_tuned = LGBMRegressor(**best_params)
lgbm_tuned.fit(X_trainval, y_trainval)
y_pred_lgbm = lgbm_tuned.predict(X_test)
final_results['LightGBM'] = regression_metrics(y_test, y_pred_lgbm)
final_models['LightGBM'] = lgbm_tuned

# --- XGBoost ---
best_params = studies['XGBoost'].best_params.copy()
best_params.update({'random_state': RANDOM_STATE, 'n_jobs': -1})
# Remove early_stopping_rounds for final retraining without eval_set
best_params.pop('early_stopping_rounds', None)
xgb_tuned = XGBRegressor(**best_params)
xgb_tuned.fit(X_trainval, y_trainval)
y_pred_xgb = xgb_tuned.predict(X_test)
final_results['XGBoost'] = regression_metrics(y_test, y_pred_xgb)
final_models['XGBoost'] = xgb_tuned

# --- CatBoost ---
best_params = studies['CatBoost'].best_params.copy()
best_params.update({'random_seed': RANDOM_STATE, 'verbose': 0})
cat_tuned = CatBoostRegressor(**best_params)
cat_tuned.fit(X_trainval, y_trainval)
y_pred_cat = cat_tuned.predict(X_test)
final_results['CatBoost'] = regression_metrics(y_test, y_pred_cat)
final_models['CatBoost'] = cat_tuned

print("Tuned models retrained on train+val and evaluated on test set.\n")
for name, metrics in final_results.items():
    print(f"{name}: R2={metrics['R2']:.4f}, RMSE={metrics['RMSE']:.2f}, "
          f"MAE={metrics['MAE']:.2f}, MAPE={metrics['MAPE']:.2f}%")

## 6. Comparison Table & Visualisation

In [ ]:
results_df = pd.DataFrame(final_results).T.sort_values('R2', ascending=False)
print("=" * 70)
print("OPTUNA-TUNED MODEL COMPARISON (Test Set)")
print("=" * 70)
print(results_df.to_string())
print()
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sorted_models = results_df.index.tolist()
colors = ['#2196F3', '#4CAF50', '#FF9800']

# R2 Score
axes[0].barh(sorted_models, results_df['R2'], color=colors[:len(sorted_models)],
             edgecolor='black', alpha=0.85)
axes[0].set_xlabel('R² Score')
axes[0].set_title('R² Score (higher is better)')
for i, v in enumerate(results_df['R2']):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=10)

# RMSE
rmse_sorted = results_df.sort_values('RMSE')
axes[1].barh(rmse_sorted.index, rmse_sorted['RMSE'],
             color=colors[:len(sorted_models)], edgecolor='black', alpha=0.85)
axes[1].set_xlabel('RMSE')
axes[1].set_title('RMSE (lower is better)')
for i, v in enumerate(rmse_sorted['RMSE']):
    axes[1].text(v + 0.3, i, f'{v:.2f}', va='center', fontsize=10)

# MAE
mae_sorted = results_df.sort_values('MAE')
axes[2].barh(mae_sorted.index, mae_sorted['MAE'],
             color=colors[:len(sorted_models)], edgecolor='black', alpha=0.85)
axes[2].set_xlabel('MAE')
axes[2].set_title('MAE (lower is better)')
for i, v in enumerate(mae_sorted['MAE']):
    axes[2].text(v + 0.2, i, f'{v:.2f}', va='center', fontsize=10)

plt.suptitle('Optuna-Tuned Model Comparison (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '25_optuna_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to {FIGURES_DIR / '25_optuna_comparison.png'}")

## 7. Save Tuned Models

In [ ]:
# Save each tuned model
for name, model in final_models.items():
    path = MODELS_TUNED / f'{name.lower()}.joblib'
    joblib.dump(model, path)
    print(f"Saved {name} -> {path}")

# Save the best overall regression model
best_name = results_df.index[0]
best_model = final_models[best_name]
best_path = MODELS_TUNED / 'best_regression.joblib'
joblib.dump(best_model, best_path)
print(f"\nBest regression model ({best_name}) saved -> {best_path}")

## 8. Save Results

In [ ]:
results_df.to_csv(DATA_PROCESSED / 'optuna_results.csv')
print(f"Results saved to {DATA_PROCESSED / 'optuna_results.csv'}")

# Final summary
print("\n" + "=" * 60)
print("OPTUNA TUNING SUMMARY")
print("=" * 60)
print(f"Trials per model: {N_OPTUNA_TRIALS}")
print(f"Best Model: {best_name}")
print(f"  R²:    {results_df.iloc[0]['R2']:.4f}")
print(f"  RMSE:  {results_df.iloc[0]['RMSE']:.2f}")
print(f"  MAE:   {results_df.iloc[0]['MAE']:.2f}")
print(f"  MAPE:  {results_df.iloc[0]['MAPE']:.2f}%")

## Summary

Optuna Bayesian hyperparameter tuning was applied to the top-3 regression models (LightGBM, CatBoost, XGBoost) using 50 trials each, maximising R² on the validation set.

Key outcomes:
- **Tuned models** were retrained on the combined train+validation set and evaluated on the held-out test set.
- **Best parameters** and final test metrics are saved for downstream use.
- **Tuned model artefacts** are persisted in `models/tuned/` as joblib files.
- The single best regression model is additionally saved as `best_regression.joblib` for the deployment pipeline.